# Clase 13 — Deep learning en el navegador con TensorFlow.js

**Objetivos de hoy**
- Entrenar un modelo sencillo directamente en el navegador.
- Explicar cómo convertir un modelo entrenado en Python para usarlo en JavaScript.
- Relacionar ambos pasos con el caso de una IA que juega al Pong.

Consulta el [capítulo 10 del manual](../manual/10_tensorflowjs.md) para la explicación completa.

## 1. Entrena un modelo directamente en el navegador

La siguiente celda renderiza HTML/JavaScript real: se ejecuta en tu navegador (dentro de esta misma página de Colab/Jupyter), entrena una regresión lineal en TensorFlow.js, y muestra el resultado. Abre la consola de desarrollador (F12) para ver más detalle si algo no se muestra.

In [ ]:
%%html
<script src="https://cdn.jsdelivr.net/npm/@tensorflow/tfjs"></script>
<div id="resultado">Entrenando en el navegador...</div>
<script>
  const modelo = tf.sequential();
  modelo.add(tf.layers.dense({units: 1, inputShape: [1]}));
  modelo.compile({optimizer: "sgd", loss: "meanSquaredError"});

  const xs = tf.tensor2d([1, 2, 3, 4], [4, 1]);
  const ys = tf.tensor2d([2, 4, 6, 8], [4, 1]);

  modelo.fit(xs, ys, {epochs: 200}).then(() => {
    const prediccion = modelo.predict(tf.tensor2d([5], [1, 1]));
    prediccion.data().then(valores => {
      document.getElementById("resultado").innerText =
        "Predicción para 5 (se espera cerca de 10): " + valores[0].toFixed(2);
    });
  });
</script>

## 2. Entrena en Python, convierte para el navegador

Entrenamos un modelo de Keras conocido (como el del capítulo 1) y lo convertimos al formato que TensorFlow.js puede cargar.

In [ ]:
import tensorflow as tf

(x_entreno, y_entreno), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_entreno, x_test = x_entreno / 255.0, x_test / 255.0

modelo = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax"),
])
modelo.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
modelo.fit(x_entreno, y_entreno, epochs=2)
modelo.save("modelo_mnist.h5")

In [ ]:
!pip install -q tensorflowjs
!tensorflowjs_converter --input_format=keras modelo_mnist.h5 modelo_web/
!ls modelo_web/

La carpeta `modelo_web/` contiene ahora un `model.json` y los pesos en formato binario. Esos archivos son los que cargarías en tu página con `tf.loadLayersModel("modelo_web/model.json")`, tal como muestra el capítulo 10 del manual.

## 3. El caso de una IA jugando al Pong

El proyecto original de esta lección entrena una red que recibe la posición de la pelota y de la pala como entrada, y decide mover la pala arriba, abajo o dejarla quieta —el mismo enfoque de aprendizaje por refuerzo del capítulo 6, aquí desplegado en el navegador en lugar de en Python—. El patrón es idéntico al que acabas de practicar: entrenar en Python, convertir el modelo, y cargarlo con `tf.loadLayersModel` en una página HTML donde el usuario puede jugar contra la IA.

## Resumen

- Entrenaste un modelo directamente en el navegador con TensorFlow.js.
- Convertiste un modelo de Keras a un formato que el navegador puede cargar con `tensorflowjs_converter`.
- Un juego con IA, como el Pong, sigue exactamente este mismo patrón de entrenar en Python y desplegar en el navegador.

Continúa con el [notebook de práctica de esta clase](../notebooks_practica/clase13_practica_tensorflowjs.ipynb).